In [ ]:
# =========================================================
# GEODESIC DECODING — CONTROLLED 6-METHOD STUDY  (Kaggle 2x T4)
# Paste into ONE Kaggle cell. Accelerator: GPU T4 x2. Internet: ON.
# Dataset: /kaggle/input/datasets/pikanigga/gsm8k-500 (questions_500.csv +
# answers_500.csv; auto-detected, columns auto-mapped).
#
# METHOD MENU (same 500 questions, same split, same prompt/parser/seed):
#   M0 greedy          argmax token                      (baseline)
#   M1 nll_only        cost = z(nll)                     (SANITY: provably == M0,
#                        because cand0 = argmax p = min NLL; kept to verify pipeline)
#   M2 fr_only         cost = z( FR(q_j, centroid(Q)) )  (geometric CONSENSUS: pick the
#                        token whose one-step future is closest to the top-k average
#                        future. NOTE: "FR to the greedy future" alone can NEVER deviate
#                        from greedy (its min is cand0, same as NLL) -> that variant is
#                        mathematically degenerate. Consensus-FR is the non-degenerate
#                        geometric selection rule. d_to_greedy is still recorded.)
#   M3 entropy_only    cost = z( H(q_j) )                (pick most confident future)
#   M4 full_geodesic   cost = z(nll) + 0.5 z(d_to_greedy) + 0.5 z(H(q_j))  (v2-style axes,
#                        z-score normalization fixes v2's min-max cand0-pinning flaw)
#   M5 conservative    full_geodesic logic but ONLY when H(p) > 2.0 nat, else greedy
#
# SPLIT POLICY (frozen BEFORE the run, saved to split_map.csv):
#   seed-42 permutation of the 500 questions -> first 100 = dev (debugging/
#   thresholds), remaining 400 = held-out test (final numbers ONLY).
#
# SPEED: 2 worker processes, one per T4 (model replicated, questions split).
#   ~500 q x 6 methods ~ 9-10 h sequential -> ~4.5-5 h here on 2x T4.
#   Quick smoke test first:  import os; os.environ["GEODESIC_LIMIT"]="8"
#   (runs 8 questions ~5 min). If the kernel dies: re-run the cell -> resumes
#   per (question, method) from part files in the output folder.
#
# OUTPUTS (/kaggle/working/geodesic_study/):
#   predictions.csv, token_traces.csv, summary.csv, summary_by_split.csv,
#   paired_comparison.csv, failure_cases.csv, split_map.csv,
#   experiment_config.json, accuracy_overview.png, failures_*.csv logs
# Then run the GRAPHS cell (cell 2) for the full figure set.
# =========================================================
import os, re, io, sys, json, time, glob, math, random, shutil, traceback
import multiprocessing as mp
import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import binomtest

# ---- shim for mixed pandas installs (idempotent) ----
import pandas.errors as _pd_errors
if not hasattr(_pd_errors, "Pandas4Warning"):
    class Pandas4Warning(Warning):
        pass
    _pd_errors.Pandas4Warning = Pandas4Warning

# ---------------- 0. CONFIG ----------------
SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 512
TOP_K = 4
ALPHA = 0.5                # weight of z(d_to_greedy) in full/conservative
BETA = 0.5                 # weight of z(H(q_j)) in full/conservative
ENT_GATE_TAU = 1.0         # nat; geometric terms engage only when H(p) > tau
CONSERVATIVE_TAU = 2.0     # nat; M5 deviates only above this
NORMALIZATION = "zscore"   # per-step, per-axis (fixes v2 min-max flaw)
DEV_SIZE = int(os.environ.get("GEODESIC_DEV_SIZE", "100"))
N_GPUS = int(os.environ.get("GEODESIC_GPUS", "2"))
FORCE_SEQ = os.environ.get("GEODESIC_FORCE_SEQ", "0") == "1"
LIMIT = int(os.environ.get("GEODESIC_LIMIT", "0"))      # cap #questions (smoke test)
N_BOOT = int(os.environ.get("GEODESIC_NBOOT", "5000"))
RESUME = True
USE_KV_CACHE = True
EMPTY_CACHE_EVERY = 25

OUT_DIR = os.environ.get("GEODESIC_OUT_DIR", "/kaggle/working/geodesic_study")
os.makedirs(OUT_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

METHODS = ["greedy", "nll_only", "fr_only", "entropy_only", "full_geodesic", "conservative"]
METHOD_COLORS = {"greedy": "#6B7280", "nll_only": "#9CA3AF", "fr_only": "#CC3311",
                 "entropy_only": "#009988", "full_geodesic": "#0077BB",
                 "conservative": "#EE7733"}

# ---------------- 1. HELPERS ----------------
def normalize_number(s):
    s = s.strip().replace(",", "").replace("$", "").replace(" ", "")
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except Exception:
        return s

def gold_answer(answer_text):
    s = str(answer_text)
    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", s)
    if m:
        return normalize_number(m.group(1))
    try:
        return normalize_number(s)          # answers file may hold bare numbers
    except Exception:
        pass
    nums = re.findall(r"([-+]?\d[\d,]*(?:\.\d+)?)", s)
    if nums:
        return normalize_number(nums[-1])
    raise ValueError("Cannot parse gold answer: " + s[:80])

def predicted_answer(text):
    t = text.replace("*", "").replace("`", "")
    m = re.findall(r"[Ff]inal\s+answer\s*[:：=]\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)", t)
    if m:
        return normalize_number(m[-1])
    m = re.findall(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", t)
    if m:
        return normalize_number(m[-1])
    nums = re.findall(r"([-+]?\d[\d,]*(?:\.\d+)?)", t)
    return normalize_number(nums[-1]) if nums else ""

def make_prompt(question):
    messages = [
        {"role": "system",
         "content": ("Solve the math problem carefully. Show your work step by step. "
                     "End your reply with exactly: Final answer: <number>.")},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True)

def entropy(p):
    p = p.float()
    return -(p * torch.log(p.clamp_min(1e-12))).sum(dim=-1)

def fisher_rao_distance(p, q):
    p = p.float(); q = q.float()
    affinity = torch.sqrt(p.clamp_min(0) * q.clamp_min(0)).sum(-1)
    affinity = affinity.clamp(0.0, 1.0)
    return 2.0 * torch.acos(affinity)

def zscore(x):
    x = x.float()
    return (x - x.mean()) / (x.std() + 1e-6)

def cache_undo(past, keep_len):
    try:
        past.crop(-1)
    except Exception:
        past.crop(keep_len)

def _append_csv(df, path):
    if os.path.exists(path):
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, mode="w", header=True, index=False)

# ---------------- 2. DECODING ENGINE ----------------
@torch.inference_mode()
def generate_one(prompt, method):
    global USE_KV_CACHE
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    ids = encoded["input_ids"].to(model.device)
    eos_id = tokenizer.eos_token_id

    use_nll = method in ("nll_only", "full_geodesic", "conservative")
    use_cent = method == "fr_only"
    use_dtax = method in ("full_geodesic", "conservative")
    use_ent = method in ("entropy_only", "full_geodesic", "conservative")
    needs_Q = use_cent or use_dtax or use_ent
    tau = 0.0 if method == "nll_only" else (
        CONSERVATIVE_TAU if method == "conservative" else ENT_GATE_TAU)

    trace = []
    forward_calls = 0
    start = time.perf_counter()
    _crop_checked = False
    stopped_on = "token_limit"
    try:
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass

    past, logits = None, None
    if USE_KV_CACHE:
        try:
            out = model(input_ids=ids, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1, :]
        except Exception:
            past = None
    if past is None:
        out = model(input_ids=ids)
        logits = out.logits[:, -1, :]
    forward_calls += 1
    seq_len = ids.shape[1]

    for step in range(MAX_NEW_TOKENS):
        p = torch.softmax(logits.float(), dim=-1)
        logp = torch.log(p.clamp_min(1e-12))
        H_p = float(entropy(p).item())

        if method == "greedy":
            chosen_id = int(torch.argmax(logits, dim=-1).item())
            fr_best, dcent_best, qH_best = float("nan"), float("nan"), float("nan")
            cost_best = float(-logp[0, chosen_id].item())
            gate = 0.0
            greedy_tok = chosen_id
            topk_ids = ""
        else:
            k = min(TOP_K, p.shape[-1])
            cand = torch.topk(p[0], k=k).indices
            nll = -logp[0, cand]
            gate = 1.0 if H_p > tau else 0.0
            active = gate == 1.0
            greedy_tok = int(cand[0].item())
            topk_ids = ";".join(str(int(c)) for c in cand.tolist())

            Q = None
            if active and needs_Q and past is not None:
                try:
                    qs = []
                    for j in range(k):
                        o = model(input_ids=cand[j].view(1, 1),
                                  past_key_values=past, use_cache=True)
                        qs.append(torch.softmax(o.logits[:, -1, :].float(), dim=-1)[0])
                        cache_undo(past, seq_len)
                        if not _crop_checked:
                            cur = (past.get_seq_length()
                                   if hasattr(past, "get_seq_length") else seq_len)
                            if int(cur) != seq_len:
                                raise RuntimeError("cache_undo semantics mismatch")
                            _crop_checked = True
                    Q = torch.stack(qs)
                    forward_calls += k
                except Exception:
                    Q = None
                    past = None
                    USE_KV_CACHE = False
                    print("[warn] KV-cache failed once -> recompute fallback.", flush=True)
            if active and needs_Q:
                if Q is None:
                    seqs = torch.cat([ids.expand(k, -1),
                                      cand.view(k, 1).to(ids.device)], dim=1)
                    o = model(input_ids=seqs)
                    Q = torch.softmax(o.logits[:, -1, :].float(), dim=-1)
                    forward_calls += 1

            if active and needs_Q:
                q_ent = entropy(Q)
                d_greedy = fisher_rao_distance(Q, Q[0:1].expand(k, -1))
                d_greedy[0] = 0.0
                centroid = Q.mean(dim=0, keepdim=True)
                d_cent = fisher_rao_distance(Q, centroid.expand(k, -1))
                if use_nll and not (use_dtax or use_ent):
                    cost = zscore(nll)                       # nll_only (== greedy)
                elif use_cent:
                    cost = zscore(d_cent)                    # fr_only (consensus)
                elif use_ent and not use_dtax:
                    cost = zscore(q_ent)                     # entropy_only
                else:
                    cost = zscore(nll) + ALPHA * zscore(d_greedy) + BETA * zscore(q_ent)
                best = int(torch.argmin(cost).item())
                fr_best = float(d_greedy[best].item())
                dcent_best = float(d_cent[best].item())
                qH_best = float(q_ent[best].item())
                cost_best = float(cost[best].item())
            elif use_nll and not needs_Q:                    # nll_only, gate-free
                cost = zscore(nll)
                best = int(torch.argmin(cost).item())
                fr_best, dcent_best, qH_best = float("nan"), float("nan"), float("nan")
                cost_best = float(cost[best].item())
            else:                                            # gate closed -> argmax p
                best = 0
                fr_best, dcent_best, qH_best = float("nan"), float("nan"), float("nan")
                cost_best = float(nll[best].item())
            chosen_id = int(cand[best].item())

        nll_chosen = float(-logp[0, chosen_id].item())
        trace.append({
            "step": step, "chosen_token": chosen_id, "greedy_token": greedy_tok,
            "nll": nll_chosen, "fr": fr_best, "d_centroid": dcent_best,
            "next_entropy": qH_best, "objective": cost_best,
            "current_entropy": H_p, "gate": gate, "topk_ids": topk_ids,
        })

        ids = torch.cat([ids, torch.tensor([[chosen_id]], device=ids.device)], dim=1)
        if chosen_id == eos_id:
            stopped_on = "eos"
            break

        if past is not None:
            try:
                out = model(input_ids=ids[:, -1:], past_key_values=past, use_cache=True)
                past, logits = out.past_key_values, out.logits[:, -1, :]
                seq_len += 1
            except Exception:
                past = None
                USE_KV_CACHE = False
        if past is None:
            out = model(input_ids=ids)
            logits = out.logits[:, -1, :]
        forward_calls += 1

    elapsed = time.perf_counter() - start
    new_tokens = ids[0, encoded["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    trace_df = pd.DataFrame(trace)
    gate_rate = float(trace_df["gate"].mean()) if not trace_df.empty else float("nan")
    gpu_mb = float("nan")
    try:
        if torch.cuda.is_available():
            gpu_mb = torch.cuda.max_memory_allocated() / 2**20
    except Exception:
        pass
    return {"text": text, "trace": trace_df, "seconds": elapsed,
            "forward_calls": forward_calls, "generated_tokens": int(len(new_tokens)),
            "gate_rate": gate_rate, "stopped_on": stopped_on, "gpu_mem_mb": gpu_mb}

# ---------------- 3. DATA + FROZEN SPLIT (parent process) ----------------
def find_input_dir():
    d = os.environ.get("GEODESIC_DATA", "/kaggle/input/datasets/pikanigga/gsm8k-500")
    if os.path.isdir(d):
        return d
    hits = glob.glob("/kaggle/input/**/questions*500*.csv", recursive=True) or \
           glob.glob("/kaggle/input/**/questions*.csv", recursive=True)
    if hits:
        print(f"[info] dataset auto-detected at {os.path.dirname(hits[0])}")
        return os.path.dirname(hits[0])
    raise FileNotFoundError(f"Dataset not found at {d}; set GEODESIC_DATA.")

def pick_col(df, names, fallback_min_chars=10):
    for c in df.columns:
        if any(n in c.lower() for n in names):
            return c
    str_cols = [c for c in df.columns if df[c].dtype == object]
    if str_cols:
        return max(str_cols, key=lambda c: df[c].astype(str).str.len().mean())
    return df.columns[0]

DATA_DIR = find_input_dir()
qdf = pd.read_csv(os.path.join(DATA_DIR, "questions_500.csv"))
adf_path = os.path.join(DATA_DIR, "answers_500.csv")
adf = pd.read_csv(adf_path) if os.path.exists(adf_path) else None
QCOL = pick_col(qdf, ["question", "problem", "input", "prompt"])
if adf is not None:
    ACOL = pick_col(adf, ["answer", "solution", "gold", "output"])
else:
    ACOL = pick_col(qdf, ["answer", "solution", "gold"])
    adf = qdf
print(f"[data] {DATA_DIR}\n  question column: {QCOL} | answer column: {ACOL}")
n = min(len(qdf), len(adf))
if len(qdf) != len(adf):
    print(f"[warn] row counts differ (q={len(qdf)}, a={len(adf)}) -> using first {n}")
QTEXTS = qdf[QCOL].astype(str).tolist()[:n]
GOLDS = [gold_answer(v) for v in adf[ACOL].astype(str).tolist()[:n]]
N_QUESTIONS = n

_rng = random.Random(SEED)
_perm = list(range(N_QUESTIONS)); _rng.shuffle(_perm)
dev_ids = _perm[:min(DEV_SIZE, N_QUESTIONS)]
test_ids = _perm[len(dev_ids):]
SPLIT = {i: ("dev" if i in set(dev_ids) else "test") for i in range(N_QUESTIONS)}
split_map = pd.DataFrame({"question_id": range(N_QUESTIONS),
                          "split": [SPLIT[i] for i in range(N_QUESTIONS)]})
split_map.to_csv(f"{OUT_DIR}/split_map.csv", index=False)
ITEMS = [(i, SPLIT[i]) for i in dev_ids] + [(i, SPLIT[i]) for i in test_ids]
if LIMIT:
    ITEMS = ITEMS[:LIMIT]
print(f"[split] frozen: {len(dev_ids)} dev + {len(test_ids)} test "
      f"(seed {SEED}; saved to split_map.csv)")
if LIMIT:
    print(f"[smoke] GEODESIC_LIMIT={LIMIT} -> running only the first {LIMIT} items")

# ---------------- 4. WORKER (runs in each GPU process, or sequentially) ----------------
def run_items(gpu_label, items, qtexts, golds):
    """Load model on this process's device, run all missing (question, method)
    pairs, append each result to its own part CSV immediately (crash-safe)."""
    global tokenizer, model, device
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_label)
    if torch.cuda.is_available():
        device = "cuda"
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        device = "cpu"
        dtype = torch.float32
    print(f"[GPU{gpu_label}] device={device} dtype={dtype} loading model...", flush=True)
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)
    model = model.to(device)
    model.eval()
    print(f"[GPU{gpu_label}] model ready.", flush=True)

    part_pred = f"{OUT_DIR}/predictions_part{gpu_label}.csv"
    part_tr = f"{OUT_DIR}/token_traces_part{gpu_label}.csv"
    part_fail = f"{OUT_DIR}/failures_part{gpu_label}.csv"
    fail_log = []
    if os.path.exists(part_fail):
        try:
            fail_log = pd.read_csv(part_fail).to_dict("records")
        except Exception:
            fail_log = []

    done = set()
    for f in glob.glob(f"{OUT_DIR}/predictions_part*.csv"):
        try:
            old = pd.read_csv(f)
            if len(old):
                done |= set(zip(old["question_id"].astype(int), old["method"]))
        except Exception:
            pass
    todo = [(q, s) for (q, s) in items
            if any((q, m) not in done for m in METHODS)]
    print(f"[GPU{gpu_label}] {len(todo)}/{len(items)} questions need work "
          f"(resume check).", flush=True)

    t0 = time.time()
    n_done = 0
    for (qid, split) in todo:
        prompt = make_prompt(qtexts[qid])
        for method in METHODS:
            if (qid, method) in done:
                continue
            try:
                r = generate_one(prompt, method)
                row = {"question_id": qid, "method": method, "split": split,
                       "question": qtexts[qid], "gold": golds[qid],
                       "prediction": predicted_answer(r["text"]),
                       "correct": int(predicted_answer(r["text"]) == golds[qid]),
                       "seconds": r["seconds"], "forward_calls": r["forward_calls"],
                       "generated_tokens": r["generated_tokens"],
                       "gate_rate": r["gate_rate"], "stopped_on": r["stopped_on"],
                       "gpu_mem_mb": r["gpu_mem_mb"], "response": r["text"]}
                _append_csv(pd.DataFrame([row]), part_pred)
                if not r["trace"].empty:
                    tr = r["trace"].copy()
                    tr["question_id"] = qid; tr["method"] = method
                    _append_csv(tr, part_tr)
            except Exception as e:
                fail_log.append({"question_id": qid, "method": method,
                                 "error": f"{type(e).__name__}: {e}",
                                 "traceback": traceback.format_exc(limit=4)})
                pd.DataFrame(fail_log).to_csv(part_fail, index=False)
                print(f"[GPU{gpu_label}][fail] q{qid}/{method}: "
                      f"{type(e).__name__}: {e}", flush=True)
        n_done += 1
        if n_done % 10 == 0 or n_done == len(todo):
            rate = (time.time() - t0) / n_done
            eta = rate * (len(todo) - n_done) / 3600
            print(f"[GPU{gpu_label}] {n_done}/{len(todo)} q | {rate:.1f} s/q | "
                  f"ETA {eta:.1f} h "
                  f"(~{time.strftime('%H:%M', time.localtime(time.time() + eta*3600))})",
                  flush=True)
        if device == "cuda" and n_done % EMPTY_CACHE_EVERY == 0:
            torch.cuda.empty_cache()
    print(f"[GPU{gpu_label}] worker finished.", flush=True)

# ---------------- 5. ORCHESTRATION ----------------
tokenizer = model = None
device = "cpu"

def read_parts(pattern):
    dfs = []
    for f in sorted(glob.glob(f"{OUT_DIR}/{pattern}")):
        try:
            df = pd.read_csv(f)
            if len(df):
                dfs.append(df)
        except Exception:
            print(f"[warn] could not read {f} (skipping)", flush=True)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

t_start = time.time()
if FORCE_SEQ or N_GPUS < 2:
    print("[run] sequential mode (one device).")
    run_items(0, ITEMS, QTEXTS, GOLDS)
else:
    try:
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(MODEL_NAME)
            print("[run] model files pre-cached (both workers can start fast).")
        except Exception as e:
            print(f"[warn] model pre-download failed ({e}). If both workers then "
                  "fail, enable Internet: Settings -> Internet ON.")
        ctx = mp.get_context("fork")
        halves = [ITEMS[0::2], ITEMS[1::2]]
        procs = [ctx.Process(target=run_items, args=(g, halves[g], QTEXTS, GOLDS))
                 for g in range(2)]
        for pr in procs:
            pr.start()
        for pr in procs:
            pr.join()
        codes = [pr.exitcode for pr in procs]
        print(f"[run] worker exit codes: {codes}")
        if any(c not in (0, None) for c in codes):
            print("[warn] a worker crashed -> sequential fallback will fill the gaps.")
    except Exception as e:
        print(f"[warn] multiprocessing unavailable ({e}) -> sequential fallback.")
    # sequential fallback fills anything the workers missed
    done_all = set()
    parts = read_parts("predictions_part*.csv")
    if len(parts):
        done_all = set(zip(parts["question_id"].astype(int), parts["method"]))
    missing = sum(1 for (q, s) in ITEMS for m in METHODS if (q, m) not in done_all)
    if missing:
        print(f"[run] {missing} (question, method) pairs left -> sequential fill.")
        run_items("seq", ITEMS, QTEXTS, GOLDS)
    else:
        print("[run] workers completed everything.")

# ---------------- 6. MERGE + TABLES ----------------
pred = read_parts("predictions_part*.csv")
if pred.empty:
    raise SystemExit("No results produced -- check failures_part*.csv and the "
                     "worker logs above (Internet ON? GPU attached?).")
pred = pred.drop_duplicates(subset=["question_id", "method"], keep="last")
pred = pred.sort_values(["question_id", "method"]).reset_index(drop=True)
pred.to_csv(f"{OUT_DIR}/predictions.csv", index=False)

tr = read_parts("token_traces_part*.csv")
if not tr.empty:
    tr = tr.drop_duplicates(subset=["question_id", "method", "step"], keep="last")
    tr = tr.sort_values(["question_id", "method", "step"]).reset_index(drop=True)
    tr.to_csv(f"{OUT_DIR}/token_traces.csv", index=False)

summary = pred.groupby("method").agg(
    accuracy=("correct", "mean"), correct=("correct", "sum"),
    examples=("correct", "count"), mean_seconds=("seconds", "mean"),
    median_seconds=("seconds", "median"),
    mean_forward_calls=("forward_calls", "mean"),
    mean_generated_tokens=("generated_tokens", "mean"),
    mean_gate_rate=("gate_rate", "mean"),
    truncation_rate=("stopped_on", lambda s: float((s != "eos").mean())),
).reset_index()
summary["accuracy_pct"] = 100 * summary["accuracy"]
summary.to_csv(f"{OUT_DIR}/summary.csv", index=False)
summary_by_split = pred.groupby(["split", "method"]).agg(
    accuracy=("correct", "mean"), correct=("correct", "sum"),
    examples=("correct", "count"), mean_seconds=("seconds", "mean"),
).reset_index()
summary_by_split["accuracy_pct"] = 100 * summary_by_split["accuracy"]
summary_by_split.to_csv(f"{OUT_DIR}/summary_by_split.csv", index=False)

wide = pred.pivot(index="question_id", columns="method", values="correct")
wide = wide.join(split_map.set_index("question_id"))
wide.to_csv(f"{OUT_DIR}/paired_comparison.csv")

def paired_stats(sub, m):
    a, b = sub[m].astype(float), sub["greedy"].astype(float)
    diff = a - b
    d = diff.mean()
    rng = np.random.default_rng(SEED)
    arr = diff.values; nb = len(arr); boots = np.empty(N_BOOT)
    for j in range(N_BOOT):
        idx = rng.integers(0, nb, size=nb)
        boots[j] = arr[idx].mean()
    wins = int(((a == 1) & (b == 0)).sum())
    losses = int(((a == 0) & (b == 1)).sum())
    pval = binomtest(wins, n=wins + losses, p=0.5).pvalue if wins + losses else 1.0
    return d, np.quantile(boots, 0.025), np.quantile(boots, 0.975), wins, losses, pval

test_wide = wide[wide["split"] == "test"]
print("\n=== MAIN RESULTS (all) ===")
print(summary.round(4).to_string(index=False))
print("\n=== BY SPLIT ===")
print(summary_by_split.round(4).to_string(index=False))
print(f"\n=== PAIRED DIFFS vs greedy (HELD-OUT TEST, n={len(test_wide)}) ===")
for m in METHODS:
    if m == "greedy" or m not in test_wide.columns:
        continue
    d, lo, hi, w, l, pv = paired_stats(test_wide, m)
    print(f"{m:>14} vs greedy: delta={100*d:+.1f}pp CI=[{100*lo:+.1f},{100*hi:+.1f}] "
          f"wins={w} losses={l} McNemar_p={pv:.4f}")

# failure / regression cases (plan Phase 5 A-D)
fc_rows = []
for m in METHODS:
    if m == "greedy" or m not in wide.columns:
        continue
    for qid in wide.index:
        g, v = wide.at[qid, "greedy"], wide.at[qid, m]
        if pd.isna(g) or pd.isna(v):
            continue
        cat = {(1, 1): "A_both_right", (1, 0): "B_regression_greedy_right",
               (0, 1): "C_improvement_greedy_wrong", (0, 0): "D_both_wrong"}[(int(g), int(v))]
        if cat != "A_both_right":
            row = pred[(pred.question_id == qid) & (pred.method == m)]
            grow = pred[(pred.question_id == qid) & (pred.method == "greedy")]
            fc_rows.append({"question_id": qid, "split": SPLIT[qid], "method": m,
                            "category": cat, "gold": GOLDS[qid],
                            "greedy_pred": (grow["prediction"].iloc[0] if len(grow) else ""),
                            "method_pred": (row["prediction"].iloc[0] if len(row) else ""),
                            "question": QTEXTS[qid]})
fc = pd.DataFrame(fc_rows)
fc.to_csv(f"{OUT_DIR}/failure_cases.csv", index=False)
print(f"\n[files] failure_cases.csv: {len(fc)} non-A rows "
      f"(B=regression, C=improvement, D=both wrong).")

config = {
    "model": MODEL_NAME, "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS,
    "top_k": TOP_K, "alpha": ALPHA, "beta": BETA,
    "ent_gate_tau": ENT_GATE_TAU, "conservative_tau": CONSERVATIVE_TAU,
    "normalization": NORMALIZATION, "n_questions": N_QUESTIONS,
    "dev_ids": sorted(dev_ids), "test_ids": sorted(test_ids),
    "methods": METHODS, "n_boot": N_BOOT, "kv_cache": USE_KV_CACHE,
    "dataset_dir": DATA_DIR, "question_col": QCOL, "answer_col": ACOL,
    "torch": torch.__version__, "pandas": pd.__version__,
    "started": time.strftime("%Y-%m-%d %H:%M:%S"),
    "finished": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with open(f"{OUT_DIR}/experiment_config.json", "w") as f:
    json.dump(config, f, indent=2)

# quick overview figure (full figure set -> cell 2)
fig, ax = plt.subplots(figsize=(8.5, 5), constrained_layout=True)
errs = []
for m in summary["method"]:
    sub = pred[pred.method == m]
    rng = np.random.default_rng(SEED)
    boots = [rng.integers(0, len(sub), len(sub)) for _ in range(500)]
    bs = [sub["correct"].values[b].mean() for b in boots]
    errs.append(100 * (np.quantile(bs, 0.975) - np.quantile(bs, 0.025)) / 2)
ax.bar(summary["method"], summary["accuracy_pct"], yerr=errs,
       color=[METHOD_COLORS[m] for m in summary["method"]], capsize=4)
ax.axhline(100 * test_wide["greedy"].mean(), color="black", ls="--", lw=1,
           label="greedy (test)")
ax.set_ylabel("Exact-match accuracy (%)")
ax.set_title(f"6-method study overview (n={pred.question_id.nunique()}; "
             "error bars = 95% bootstrap CI)")
ax.legend(); ax.set_ylim(0, 100)
fig.savefig(f"{OUT_DIR}/accuracy_overview.png", dpi=200)
plt.show()

print("\nSaved in:", OUT_DIR)
zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("All results zipped:", zip_path, "<- download from the Output panel.")
print("\nNEXT: run the GRAPHS cell (cell 2) for the full nuclear figure set.")

In [ ]:
# =========================================================
# GEODESIC STUDY — NUCLEAR GRAPHS + STATS  (Kaggle, ZERO GPU)
# Paste into the NEXT cell after the study run finishes.
# Reads predictions.csv + token_traces.csv from the study folder,
# computes the plan's Phase-4 metrics, and renders 8 presentation-grade
# figures into <OUT_DIR>/figures/. Rerun any time (cheap, no GPU).
# All inferential stats use the HELD-OUT TEST split only (per the plan);
# dev split appears only in the accuracy comparison figure.
# =========================================================
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata, binomtest

import pandas.errors as _pd_errors
if not hasattr(_pd_errors, "Pandas4Warning"):
    class Pandas4Warning(Warning):
        pass
    _pd_errors.Pandas4Warning = Pandas4Warning

OUT_DIR = os.environ.get("GEODESIC_OUT_DIR", "/kaggle/working/geodesic_study")
FIG_DIR = os.path.join(OUT_DIR, "figures"); os.makedirs(FIG_DIR, exist_ok=True)
N_BOOT = int(os.environ.get("GEODESIC_NBOOT2", "1500"))
SEED = 42
METHODS = ["greedy", "nll_only", "fr_only", "entropy_only", "full_geodesic", "conservative"]
METHOD_COLORS = {"greedy": "#6B7280", "nll_only": "#9CA3AF", "fr_only": "#CC3311",
                 "entropy_only": "#009988", "full_geodesic": "#0077BB",
                 "conservative": "#EE7733"}
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.titleweight": "bold"})

pred = pd.read_csv(f"{OUT_DIR}/predictions.csv")
tr = pd.read_csv(f"{OUT_DIR}/token_traces.csv")
try:
    split_map = pd.read_csv(f"{OUT_DIR}/split_map.csv")
    SPLIT = dict(zip(split_map.question_id.astype(int), split_map.split))
except Exception:
    SPLIT = dict(zip(pred.question_id.astype(int), pred.get("split", "test")))
TEST_Q = {q for q, s in SPLIT.items() if s == "test"}
print(f"Loaded predictions {pred.shape}, traces {tr.shape}; "
      f"test questions: {len(TEST_Q)}")

# ---------------- features per (method, question) ----------------
f = tr.groupby(["method", "question_id"]).agg(
    n_steps=("step", "count"),
    mean_H=("current_entropy", "mean"),
    max_H=("current_entropy", "max"),
    late_H=("current_entropy", lambda s: s.tail(max(1, len(s) // 4)).mean()),
    mean_nll=("nll", "mean"),
    gate_frac=("gate", "mean"),
    mean_next_H=("next_entropy", "mean"),
    mean_fr=("fr", "mean"),
    mean_dcent=("d_centroid", "mean") if "d_centroid" in tr.columns else ("fr", "mean"),
).reset_index()
feat = f.merge(pred[["question_id", "method", "correct"]], on=["method", "question_id"])
feat = feat[feat.question_id.isin(TEST_Q)].copy()

# COMBINED confidence = mean percentile-rank of risk features (within method),
# inverted so higher = more likely correct. NaN (e.g. greedy has no next_entropy
# / gate) -> neutral 0.5, so every method gets a valid score.
def _pct_risk(s):
    s = np.asarray(s, float)
    out = np.full(len(s), 0.5)
    m = ~np.isnan(s)
    if m.sum() > 1:
        r = rankdata(s[m])
        span = r.max() - r.min()
        out[m] = (r - 1) / span if span > 0 else 0.5
    return out

risk_parts = [_pct_risk(feat["mean_H"]), _pct_risk(feat["max_H"]),
              _pct_risk(feat["mean_next_H"]), _pct_risk(feat["gate_frac"])]
feat["conf_comb"] = 1.0 - np.mean([p for p in risk_parts], axis=0)
feat["conf_meanH"] = -feat["mean_H"]
feat["conf_maxH"] = -feat["max_H"]
feat["conf_nextH"] = -feat["mean_next_H"]

def auroc(y, score):
    y = np.asarray(y, float); s = np.asarray(score, float)
    m = ~np.isnan(s) & ~np.isnan(y)
    y, s = y[m], s[m]
    if len(np.unique(y)) < 2 or len(y) < 4:
        return np.nan
    r = rankdata(s)
    n1 = int(y.sum()); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n0 * n1)

def auroc_ci(y, score, n_boot=N_BOOT):
    y = np.asarray(y, float); s = np.asarray(score, float)
    m = ~np.isnan(s) & ~np.isnan(y); y, s = y[m], s[m]
    if len(np.unique(y)) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(SEED); vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        a = auroc(y[idx], s[idx])
        if not np.isnan(a):
            vals.append(a)
    if not vals:
        return np.nan, np.nan
    return float(np.quantile(vals, .025)), float(np.quantile(vals, .975))

def auprc(y, score):
    y = np.asarray(y, float); s = np.asarray(score, float)
    m = ~np.isnan(s); y, s = y[m], s[m]
    order = np.argsort(-s); y = y[order]
    tp = np.cumsum(y); prec = tp / (np.arange(len(y)) + 1)
    rec = tp / max(y.sum(), 1)
    d = np.diff(np.concatenate([[0], rec]))
    return float((d * prec).sum())

def brier(y, score):
    y = np.asarray(y, float); s = np.asarray(score, float)
    m = ~np.isnan(s); y, s = y[m], s[m]
    if not len(y):
        return np.nan
    lo, hi = np.nanmin(s), np.nanmax(s)
    c = (s - lo) / (hi - lo) if hi > lo else s * 0 + 0.5
    return float(np.mean((c - y) ** 2))

FEATURES = [("conf_meanH", "mean_H"), ("conf_maxH", "max_H"), ("late_H", "late_H"),
            ("mean_nll", "mean_nll"), ("gate_frac", "gate_frac"),
            ("conf_nextH", "mean_next_H"), ("conf_comb", "COMBINED")]

# ---------------- [1] accuracy + CI ----------------
wide = pred.pivot(index="question_id", columns="method", values="correct")
wide = wide.join(pd.Series(SPLIT, name="split"))
test = wide[wide.split == "test"]
rng = np.random.default_rng(SEED)
def acc_ci(sub, n_boot=2000):
    v = sub.astype(float).values
    bs = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n_boot)]
    return float(np.quantile(bs, .025)), float(np.quantile(bs, .975))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, constrained_layout=True)
for ax, sp, title in [(axes[0], "dev", "Development (n=%d)"),
                      (axes[1], "test", "Held-out TEST (n=%d)")]:
    sub = wide[wide.split == sp]
    n_sp = len(sub)
    ms = [m for m in METHODS if m in sub.columns]
    accs, errs = [], []
    for m in ms:
        lo, hi = acc_ci(sub[m])
        a = 100 * sub[m].mean()
        accs.append(a)
        errs.append([a - 100 * lo, 100 * hi - a])
    ax.bar(ms, accs, yerr=np.array(errs).T,
           color=[METHOD_COLORS[m] for m in ms], capsize=4)
    ax.set_title(title % n_sp)
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylim(0, 100)
    for i, a in enumerate(accs):
        ax.text(i, a + 4, f"{a:.1f}", ha="center", fontsize=9)
axes[0].set_ylabel("Exact-match accuracy (%)")
fig.suptitle("Accuracy by method (95% bootstrap CI)", fontweight="bold")
fig.savefig(f"{FIG_DIR}/fig1_accuracy_ci.png", dpi=200)
plt.show()
print("FIG 1 -> fig1_accuracy_ci.png | Accuracy per method, dev vs test. The test panel is the final comparison.")

# ---------------- [2] paired deltas forest (test) ----------------
rows = []
for m in METHODS:
    if m == "greedy" or m not in test.columns:
        continue
    a, b = test[m].astype(float), test["greedy"].astype(float)
    d = (a - b).values
    boot = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(N_BOOT)]
    lo, hi = np.quantile(boot, .025), np.quantile(boot, .975)
    w = int(((a == 1) & (b == 0)).sum()); l = int(((a == 0) & (b == 1)).sum())
    pv = binomtest(w, n=w + l, p=.5).pvalue if w + l else 1.0
    rows.append({"method": m, "delta_pp": 100 * d.mean(), "lo": 100 * lo,
                 "hi": 100 * hi, "wins": w, "losses": l, "mcnemar_p": pv})
paired = pd.DataFrame(rows)
paired.to_csv(f"{OUT_DIR}/study_paired_stats.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for i, r in paired.iterrows():
    sig = (r.lo > 0) or (r.hi < 0)
    ax.errorbar(r.delta_pp, i, xerr=[[r.delta_pp - r.lo], [r.hi - r.delta_pp]],
                fmt="o", ms=10, color=METHOD_COLORS[r.method],
                alpha=1 if sig else 0.45, capsize=4, lw=2)
    ax.text(r.delta_pp, i - 0.28, f"p={r.mcnemar_p:.3f}", ha="center", fontsize=8)
ax.axvline(0, color="black", lw=1)
ax.set_yticks(range(len(paired)), paired["method"])
ax.invert_yaxis()
ax.set_xlabel("Accuracy delta vs greedy (pp) — held-out test")
ax.set_title("Paired accuracy difference (95% CI; faded = not significant)")
fig.savefig(f"{FIG_DIR}/fig2_paired_forest.png", dpi=200)
plt.show()
print("FIG 2 -> fig2_paired_forest.png | Distance from greedy, per method. If a CI touches zero, that difference is not proven.")

# ---------------- [3] outcome matrix A/B/C/D (test) ----------------
oc_rows = []
for m in METHODS:
    if m == "greedy" or m not in test.columns:
        continue
    a, b = test[m].astype(int), test["greedy"].astype(int)
    oc_rows.append({"method": m,
                    "A_both_right": int(((a == 1) & (b == 1)).sum()),
                    "B_regression": int(((a == 0) & (b == 1)).sum()),
                    "C_improvement": int(((a == 1) & (b == 0)).sum()),
                    "D_both_wrong": int(((a == 0) & (b == 0)).sum())})
oc = pd.DataFrame(oc_rows)
oc.to_csv(f"{OUT_DIR}/study_outcomes.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
cats = ["A_both_right", "B_regression", "C_improvement", "D_both_wrong"]
cat_colors = {"A_both_right": "#B0B0B0", "B_regression": "#D62728",
              "C_improvement": "#2CA02C", "D_both_wrong": "#E8E8E8"}
left = np.zeros(len(oc))
for c in cats:
    ax.barh(oc["method"], oc[c], left=left, color=cat_colors[c], label=c)
    for i, v in enumerate(oc[c]):
        if v > 3:
            ax.text(left[i] + v / 2, i, str(v), ha="center", va="center", fontsize=9)
    left += oc[c].values
ax.invert_yaxis()
ax.set_xlabel("Questions (held-out test)")
ax.set_title("Outcome matrix vs greedy: red = regression, green = improvement")
fig.legend(ncol=4, fontsize=8, loc="outside lower center")
fig.savefig(f"{FIG_DIR}/fig3_outcome_matrix.png", dpi=200)
plt.show()
print("FIG 3 -> fig3_outcome_matrix.png | Red (B) = greedy was right and the method broke it; green (C) = a broken answer fixed to right.")

# ---------------- [4] risk-coverage (test, COMBINED) ----------------
fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
sel_rows = []
for m in METHODS:
    sub = feat[feat.method == m].sort_values("conf_comb", ascending=False)
    if sub.empty:
        continue
    acc = sub["correct"].astype(float).cumsum() / (np.arange(len(sub)) + 1)
    cov = (np.arange(len(sub)) + 1) / len(sub)
    ax.plot(100 * cov, 100 * acc, color=METHOD_COLORS[m], lw=2, label=m)
    def _acc_at(cov_target, sub=sub):
        k = max(1, int(round(cov_target * len(sub))))
        return 100 * sub["correct"].astype(float).iloc[:k].mean()
    a100 = _acc_at(1.0); a80 = _acc_at(0.8)
    boot = []
    vals = sub["correct"].astype(float).values
    con = sub["conf_comb"].values
    for _ in range(N_BOOT):
        idx = rng.integers(0, len(vals), len(vals))
        v, c = vals[idx], con[idx]
        o = np.argsort(-c); v = v[o]
        k = max(1, int(round(0.8 * len(v))))
        boot.append(100 * v[:k].mean() - 100 * v.mean())
    lo, hi = np.quantile(boot, .025), np.quantile(boot, .975)
    sel_rows.append({"method": m, "acc100": a100, "acc80": a80,
                     "gain80_pp": a80 - a100, "gain_lo": lo, "gain_hi": hi})
ax.axvline(80, color="black", ls="--", lw=1)
ax.set_xlabel("Coverage (%) — most confident questions answered first")
ax.set_ylabel("Accuracy among answered (%)")
ax.set_title("Risk–coverage: abstention value of each method (test)")
ax.legend(fontsize=9); ax.set_ylim(30, 100)
fig.savefig(f"{FIG_DIR}/fig4_risk_coverage.png", dpi=200)
plt.show()
sel = pd.DataFrame(sel_rows)
sel.to_csv(f"{OUT_DIR}/study_selective.csv", index=False)
print("FIG 4 -> fig4_risk_coverage.png | Further left = more correct answers with fewer questions taken. Compare methods at the 80 percent line.")
print(sel.round(1).to_string(index=False))

# ---------------- [5] AUROC bars (test) ----------------
au_rows = []
for m in METHODS:
    sub = feat[feat.method == m]
    if sub.empty:
        continue
    y = sub["correct"].astype(float).values
    for conf_col, label in FEATURES:
        a = auroc(y, sub[conf_col].values)
        lo, hi = auroc_ci(y, sub[conf_col].values)
        au_rows.append({"method": m, "feature": label, "auroc": a,
                        "lo": lo, "hi": hi,
                        "auprc": auprc(y, sub[conf_col].values),
                        "brier": brier(y, sub[conf_col].values)})
au = pd.DataFrame(au_rows)
au.to_csv(f"{OUT_DIR}/study_auroc.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5.5), constrained_layout=True)
feats = [lb for _, lb in FEATURES]
w = 0.2
for j, m in enumerate(METHODS):
    sub = au[(au.method == m) & (au.feature.isin(feats))]
    sub = sub.set_index("feature").reindex(feats).reset_index()
    pos = np.arange(len(feats)) + (j - 1.5) * w
    ax.bar(pos, sub["auroc"], width=w, color=METHOD_COLORS[m], label=m,
           yerr=[sub["auroc"] - sub["lo"], sub["hi"] - sub["auroc"]],
           capsize=2, error_kw={"lw": 1, "alpha": 0.6})
ax.axhline(0.5, color="black", ls="--", lw=1, label="chance")
ax.set_xticks(np.arange(len(feats)), feats)
ax.set_ylabel("AUROC (confidence -> correct)")
ax.set_title("Which uncertainty feature predicts correctness? (test, 95% CI)")
fig.legend(fontsize=8, ncol=7, loc="outside lower center")
ax.set_ylim(0, 1)
fig.savefig(f"{FIG_DIR}/fig5_auroc_bars.png", dpi=200)
plt.show()
print("FIG 5 -> fig5_auroc_bars.png | 0.5 = coin flip; higher is a better monitor. Focus on the COMBINED group.")

# ---------------- [6] geometry vs correctness (test) ----------------
look = feat[feat.method.isin(["fr_only", "entropy_only", "full_geodesic", "conservative"])]
panels = [("fr", "FR to greedy future (rad)"), ("dcent", "FR to consensus (rad)"),
          ("mean_next_H", "next-step entropy (nat)"), ("mean_H", "current entropy (nat)")]
panels = [(c, t) for c, t in panels if c in look.columns]
fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4.6),
                         constrained_layout=True)
if len(panels) == 1:
    axes = [axes]
for ax, (col, title) in zip(axes, panels):
    data = [look.loc[look.correct == 1, col].dropna(),
            look.loc[look.correct == 0, col].dropna()]
    bp = ax.boxplot(data, tick_labels=["correct", "wrong"], showfliers=False,
                    patch_artist=True, widths=0.55)
    for box, col_ in zip(bp["boxes"], ["#2CA02C", "#D62728"]):
        box.set_facecolor(col_); box.set_alpha(0.5)
    ax.set_title(title)
fig.suptitle("Geometry/uncertainty by final correctness (lookahead methods, test)",
             fontweight="bold")
fig.savefig(f"{FIG_DIR}/fig6_geometry_vs_correctness.png", dpi=200)
plt.show()
print("FIG 6 -> fig6_geometry_vs_correctness.png | If the red (wrong) box stays on top, uncertainty flags errors early.")

# ---------------- [7] efficiency frontier (test) ----------------
eff_rows = []
sec_greedy = pred[(pred.method == "greedy") & (pred.question_id.isin(TEST_Q))]["seconds"].mean()
acc_greedy = 100 * test["greedy"].mean()
for m in METHODS:
    if m == "greedy":
        continue
    sub = pred[(pred.method == m) & (pred.question_id.isin(TEST_Q))]
    if sub.empty:
        continue
    eff_rows.append({"method": m, "runtime_ratio": sub["seconds"].mean() / sec_greedy,
                     "forward_ratio": sub["forward_calls"].mean() /
                                      pred[(pred.method == "greedy") &
                                           (pred.question_id.isin(TEST_Q))]["forward_calls"].mean(),
                     "delta_acc_pp": 100 * test[m].mean() - acc_greedy})
eff = pd.DataFrame(eff_rows)
eff.to_csv(f"{OUT_DIR}/study_efficiency.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
ax.axvline(1, color="black", lw=1); ax.axhline(0, color="black", lw=1)
ax.fill_betweenx([-10, 10], 0.8, 1, color="#2CA02C", alpha=0.06)
ax.fill_betweenx([-10, 10], 1, 8, color="#D62728", alpha=0.06)
for _, r in eff.iterrows():
    ax.scatter(r.runtime_ratio, r.delta_acc_pp, s=220,
               color=METHOD_COLORS[r.method], zorder=3)
    ax.annotate(f"{r.method}\n{r.runtime_ratio:.1f}x, {r.delta_acc_pp:+.1f}pp",
                (r.runtime_ratio, r.delta_acc_pp),
                textcoords="offset points", xytext=(10, 6), fontsize=9)
ax.set_xlim(0.8, max(2.2, eff.runtime_ratio.max() * 1.15))
ax.set_ylim(min(-6, eff.delta_acc_pp.min() - 2), max(6, eff.delta_acc_pp.max() + 2))
ax.set_xlabel("Runtime vs greedy (x) — held-out test")
ax.set_ylabel("Accuracy delta vs greedy (pp)")
ax.set_title("Is the extra compute worth it? (red zone = costlier AND worse)")
fig.savefig(f"{FIG_DIR}/fig7_efficiency.png", dpi=200)
plt.show()
print("FIG 7 -> fig7_efficiency.png | Methods in the red zone cost more time for less accuracy - the 'not worth it' region.")

# ---------------- [8] first-divergence vs greedy (test) ----------------
fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
div_stats = []
g_toks = tr[tr.method == "greedy"].set_index(["question_id", "step"])["chosen_token"]
for m in ["fr_only", "entropy_only", "full_geodesic", "conservative"]:
    sub = tr[(tr.method == m) & (tr.question_id.isin(TEST_Q))]
    if sub.empty:
        continue
    m_toks = sub.set_index(["question_id", "step"])["chosen_token"]
    j = pd.concat([g_toks.rename("g"), m_toks.rename("m")], axis=1).dropna()
    mism = j[j.g != j.m]
    first = (mism.reset_index().groupby("question_id")["step"].min()
             if len(mism) else pd.Series(dtype=float))
    first_vals = np.asarray(first).ravel()
    flips = oc.set_index("method").loc[m, ["B_regression", "C_improvement"]].sum()
    div_stats.append({"method": m, "diverged": int(first.index.nunique()),
                      "compared": int(j.index.get_level_values(0).nunique()),
                      "median_first_div_step": float(np.median(first_vals)) if len(first_vals) else np.nan,
                      "answer_flips": int(flips)})
    if len(first_vals):
        ax.hist(first_vals, bins=30, alpha=0.45, color=METHOD_COLORS[m], label=m)
ax.set_xlabel("First token divergence step vs greedy")
ax.set_ylabel("Question count")
ax.set_title("Where do lookahead methods leave the greedy path? (test)")
ax.legend(fontsize=9)
fig.savefig(f"{FIG_DIR}/fig8_divergence.png", dpi=200)
plt.show()
pd.DataFrame(div_stats).to_csv(f"{OUT_DIR}/study_divergence.csv", index=False)
print("FIG 8 -> fig8_divergence.png | Which step the paths split at; more mass on the right means late divergence.")

print("\n=== ALL FIGURES + TABLES SAVED ===")
print("Figures:", FIG_DIR)
print("Tables: study_paired_stats / study_selective / study_auroc / study_outcomes /",
      "study_efficiency / study_divergence .csv (in", OUT_DIR + ")")
au_p = au[au.feature == "COMBINED"].sort_values("auroc", ascending=False)
print("\nBEST MONITORS (COMBINED AUROC, test):")
print(au_p[["method", "auroc", "lo", "hi", "auprc"]].round(3).to_string(index=False))

In [ ]:
# =========================================================
# RUN 2 CELL - FULL-COVERAGE GREEDY-ANCHORED MONITOR (final)
# Same code as analysis/monitor_run.py (single source of truth).
# Full run (~3-4 h, 2x T4):  just run this cell as-is.
# Smoke test:  import os
#              os.environ['GEODESIC2_LIMIT']='2'
#              os.environ['GEODESIC2_MAX_NEW_TOKENS']='48'
#              os.environ['GEODESIC2_SEQ']='1'
#              then run this cell.
# Outputs -> /kaggle/working/geodesic_study2/monitor_*.csv
# =========================================================
# =========================================================================
# RUN 2 — FULL-COVERAGE GREEDY-ANCHORED MONITOR (geodesic Phase-2 asset)
# Single source of truth. Used as: (a) local CPU smoke test, (b) the new
# appended cell of hellio.ipynb, (c) the Kaggle kernel body for the full run.
#
# Protocol (pre-registered in experiment_protocol.md, run-2 addendum):
#   * Decoding = greedy with DETERMINISTIC tie-breaking:
#       highest p; if fp32 tie, smaller token id. (Fixes audited §3.5.)
#   * At EVERY step, for ALL k=4 candidates (no entropy gate — removes
#     audited gate-censoring §3.3): record NLL, q_j entropy,
#     FR(q_j -> q_greedy), FR(q_j -> consensus), p_value.
#   * Chosen token == candidate rank 0 always (the trajectory IS greedy).
#   * Frozen run-1 split/golds (never re-derived -> no new contamination).
# Env knobs: GEODESIC2_MODEL, GEODESIC2_LIMIT, GEODESIC2_MAX_NEW_TOKENS,
#            GEODESIC2_OUT_DIR, GEODESIC2_DATA, GEODESIC2_EMBED, GEODESIC2_SEQ
# ========================================================================
import json
import math
import os
import random
import re
import time
import traceback

import numpy as np
import pandas as pd
import torch

try:  # rarely needed; glob used for pristine check
    from glob import glob
except Exception:  # pragma: no cover
    glob = None

CONFIG = {
    "model": os.environ.get("GEODESIC2_MODEL", "Qwen/Qwen2.5-1.5B-Instruct"),
    "seed": 42,
    "top_k": 4,
    "max_new_tokens": int(os.environ.get("GEODESIC2_MAX_NEW_TOKENS", "512")),
    "n_questions_limit": int(os.environ.get("GEODESIC2_LIMIT", "500")),
    "split_source": "run-1 frozen split (geodesic_study/split_map.csv / embedded)",
    "gold_source": "run-1 golds (geodesic_study/predictions.csv / embedded)",
    "torch": torch.__version__,
}
OUT_DIR = os.environ.get("GEODESIC2_OUT_DIR", "geodesic_study2")
os.makedirs(OUT_DIR, exist_ok=True)

if glob is None:
    def glob(pat):
        return [pat]

# ------------------------------------------------------------ helpers ----
def normalize_number(s):
    s = str(s).strip().replace(",", "").replace("$", "").replace(" ", "")
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except Exception:
        return s


def predicted_answer(text):
    t = text.replace("*", "").replace("`", "")
    m = re.findall(r"[Ff]inal\s+answer\s*[:：=]\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)", t)
    if m:
        return normalize_number(m[-1])
    m = re.findall(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", t)
    if m:
        return normalize_number(m[-1])
    nums = re.findall(r"([-+]?\d[\d,]*(?:\.\d+)?)", t)
    return normalize_number(nums[-1]) if nums else ""


def entropy(p):
    return -(p * torch.log(p.clamp_min(1e-12))).sum(dim=-1)


def fisher_rao(p, q):
    aff = torch.sqrt(p.clamp_min(0) * q.clamp_min(0)).sum(-1)
    return 2.0 * torch.acos(aff.clamp(0.0, 1.0))


def topk_deterministic(p_row, k):
    """Highest-prob k tokens; ties (exact fp32 equality) -> smaller token id."""
    cand = torch.topk(p_row, k=k).indices.tolist()
    cand.sort(key=lambda i: (-float(p_row[i]), i))
    return cand


# ------------------------------------------------------------- engine ----
@torch.inference_mode()
def generate_monitor(prompt, tokenizer, model, eos_id):
    device = next(model.parameters()).device
    k = CONFIG["top_k"]
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    ids = enc["input_ids"].to(device)
    out = model(input_ids=ids, use_cache=True)
    past, logits = out.past_key_values, out.logits[:, -1, :]
    forward_calls = 1
    seq_len = ids.shape[1]
    crop_ok = []
    steps, cands = [], []
    stopped = "token_limit"
    t0 = time.perf_counter()
    for step in range(CONFIG["max_new_tokens"]):
        p = torch.softmax(logits.float(), dim=-1)[0]
        cand = torch.as_tensor(topk_deterministic(p, k), device=device)
        p_top = p[cand]
        nll = (-torch.log(p.clamp_min(1e-12)))[cand]
        H_p = float(entropy(p).item())
        p_gap = float(p_top[0].item() - p_top[1].item())

        # ---- lookahead: all k candidates, no gate ----
        qs = []
        for j in range(k):
            o = model(input_ids=cand[j].view(1, 1),
                      past_key_values=past, use_cache=True)
            qs.append(torch.softmax(o.logits[:, -1, :].float(), dim=-1)[0])
            forward_calls += 1
            before = int(past.get_seq_length())
            past.crop(-1)
            if j == 0:
                crop_ok.append(int(past.get_seq_length()) == seq_len)
        crop_semantics = "OK" if all(crop_ok) else "FAILED"
        assert crop_ok and crop_semantics == "OK", (
            f"KV crop failed at step {step}: after={past.get_seq_length()} expect={seq_len}")
        Q = torch.stack(qs)                                   # [k, V]
        q_ent = entropy(Q)                                    # [k]
        fr_greedy = fisher_rao(Q, Q[0:1].expand(k, -1))       # [k], q0 ~ 0
        centroid = Q.mean(dim=0, keepdim=True)
        fr_cons = fisher_rao(Q, centroid.expand(k, -1))       # [k]
        chosen = int(cand[0].item())
        for j in range(k):
            cands.append({
                "step": step, "rank": j,
                "token_id": int(cand[j].item()),
                "p_value": float(p_top[j].item()),
                "nll": float(nll[j].item()),
                "q_entropy": float(q_ent[j].item()),
                "fr_greedy": float(fr_greedy[j].item()),
                "fr_consensus": float(fr_cons[j].item()),
                "is_greedy": int(j == 0),
            })
        steps.append({
            "step": step, "chosen_token": chosen,
            "h_p": H_p, "nll_greedy": float(nll[0].item()),
            "p_gap01": p_gap, "tie_flag": int(p_top[0].item() == p_top[1].item()),
        })
        if chosen == eos_id:
            stopped = "eos"
            break
        ids = torch.cat([ids, cand[0:1].view(-1, 1)], dim=1)
        try:
            out = model(input_ids=ids[:, -1:],
                        past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1, :]
            forward_calls += 1
            seq_len += 1
        except Exception:
            out = model(input_ids=ids, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1, :]
            forward_calls += 1
    new_tokens = ids[0, enc["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return (text, pd.DataFrame(steps), pd.DataFrame(cands), stopped,
            time.perf_counter() - t0, forward_calls)


# --------------------------------------------------------------- main ----
def load_items():
    """Frozen split-from-run-1 questions + golds. Locally: read the study CSVs;
    on Kaggle: an embedded JSON file (GEODESIC2_EMBED) so no run-1 artifacts."""
    embed = os.environ.get("GEODESIC2_EMBED")
    if embed:
        with open(embed, encoding="utf-8") as f:
            payload = json.load(f)
        src = pd.DataFrame(payload["rows"])
    else:
        src = pd.read_csv(os.environ.get(
            "GEODESIC2_DATA", "geodesic_study/predictions.csv"),
            usecols=["question_id", "method", "question", "gold"])
        src = src[src.method == "greedy"]
    src = src.drop_duplicates("question_id").sort_values("question_id")
    src["gold"] = src.gold.map(normalize_number)
    if "split" not in src.columns:
        split_map = pd.read_csv("geodesic_study/split_map.csv")
        src = src.merge(split_map, on="question_id", how="left")
    return src.reset_index(drop=True)


def run_worker(g, num_workers, src):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    device = f"cuda:{g}" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device.startswith("cuda") else torch.float32
    try:
        tok = AutoTokenizer.from_pretrained(CONFIG["model"])
        try:
            model = AutoModelForCausalLM.from_pretrained(
                CONFIG["model"], dtype=dtype)
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(
                CONFIG["model"], torch_dtype=dtype)
        model = model.to(device)
    except Exception as e:
        pd.DataFrame([{"worker": g, "error": f"{type(e).__name__}: {e}"}]
                     ).to_csv(f"{OUT_DIR}/failures_worker{g}.csv", index=False)
        raise
    model.eval()
    eos = tok.eos_token_id
    print(f"[worker{g}] device={device} dtype={dtype}", flush=True)

    part_ans = f"{OUT_DIR}/monitor_answers_part{g}.csv"
    part_stp = f"{OUT_DIR}/monitor_steps_part{g}.csv"
    part_cnd = f"{OUT_DIR}/monitor_candidates_part{g}.csv"

    done = set()
    if os.path.exists(part_ans):
        done |= set(pd.read_csv(part_ans, usecols=["question_id"])
                    .question_id.astype(int))
    todo = [i for i in range(len(src))
            if int(src.question_id[i]) not in done and i % num_workers == g]
    for n_i, i in enumerate(todo):
        qid, qtext, gold = (int(src.question_id[i]), src.question[i],
                            src.gold[i])
        splitlab = (src.split[i] if "split" in src.columns
                    else ("dev" if i < CONFIG.get("DEV_SIZE", 100) else "test"))
        msgs = [{"role": "system", "content":
                 "Solve the math problem carefully. Show your work step by step. "
                 "End your reply with exactly: Final answer: <number>."},
                {"role": "user", "content": qtext}]
        prompt = tok.apply_chat_template(msgs, tokenize=False,
                                         add_generation_prompt=True)
        try:
            text, steps, cands, stopped, secs, fwd = generate_monitor(
                prompt, tok, model, eos)
            row = {"question_id": qid, "split": splitlab,
                   "prediction": predicted_answer(text), "gold": gold,
                   "correct": int(predicted_answer(text) == gold),
                   "stopped_on": stopped, "seconds": secs,
                   "forward_calls": fwd, "generated_tokens": len(steps),
                   "model": CONFIG["model"], "seed": 42, "response": text}
            for path, df in ((part_ans, pd.DataFrame([row])),
                             (part_stp, steps.assign(question_id=qid,
                                                     split=splitlab)),
                             (part_cnd, cands.assign(question_id=qid))):
                df.to_csv(path, mode="a",
                          header=not os.path.exists(path), index=False)
        except Exception as e:
            pd.DataFrame([{"question_id": qid,
                           "error": f"{type(e).__name__}: {e}",
                           "traceback": traceback.format_exc(limit=3)}]
                         ).to_csv(f"{OUT_DIR}/failures.csv", mode="a",
                                  header=not os.path.exists(
                                      f"{OUT_DIR}/failures.csv"), index=False)
            print(f"[fail] q{qid}: {type(e).__name__}: {e}", flush=True)
            continue
        if (n_i + 1) % 10 == 0:
            print(f"[worker{g}] {n_i + 1}/{len(todo)} done", flush=True)
        if device.startswith("cuda") and (n_i + 1) % 25 == 0:
            torch.cuda.empty_cache()


def out_dir_pristine():
    return not os.path.exists(f"{OUT_DIR}/monitor_answers_part1.csv")


def main():
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    src = load_items().head(CONFIG["n_questions_limit"]).reset_index(drop=True)
    print(f"[setup] model={CONFIG['model']} questions={len(src)} "
          f"max_new={CONFIG['max_new_tokens']}", flush=True)

    import multiprocessing as mp
    n_gpu = torch.cuda.device_count()
    t0 = time.time()
    if n_gpu >= 2 and out_dir_pristine() \
            and os.environ.get("GEODESIC2_SEQ", "0") != "1":
        ctx = mp.get_context("fork")
        procs = [ctx.Process(target=run_worker, args=(g, 2, src))
                 for g in range(2)]
        for pr in procs:
            pr.start()
        for pr in procs:
            pr.join()
    else:
        run_worker(0, 1, src)
    pd.DataFrame([{**CONFIG, "n_gpus": n_gpu,
                   "transformers": __import__("transformers").__version__,
                   "wall_seconds": time.time() - t0}]
                 ).to_csv(f"{OUT_DIR}/monitor_config.csv", index=False)
    print(f"[done] wall={(time.time() - t0) / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()
